# step_renderer

> Composable render functions for the alignment card stack column

In [ ]:
#| default_exp components.step_renderer

In [ ]:
#| export
from typing import Any, List, Optional

from fasthtml.common import Div, Span

# DaisyUI components
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, h, min_h
from cjm_fasthtml_tailwind.utilities.typography import font_size
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, justify, items, gap, grow
)
from cjm_fasthtml_tailwind.core.base import combine_classes

# Card stack library
from cjm_fasthtml_card_stack.components.viewport import render_viewport
from cjm_fasthtml_card_stack.components.progress import render_progress_indicator
from cjm_fasthtml_card_stack.core.models import CardStackState
from cjm_fasthtml_card_stack.core.constants import DEFAULT_VISIBLE_COUNT, DEFAULT_CARD_WIDTH
from cjm_fasthtml_card_stack.keyboard.actions import render_card_stack_action_buttons

# Web Audio library
from cjm_fasthtml_web_audio.components import render_audio_urls_input, render_web_audio_script
from cjm_fasthtml_web_audio.models import WebAudioConfig

# Alignment configuration (same package)
from cjm_transcript_vad_align.components.card_stack_config import (
    ALIGN_CS_CONFIG, ALIGN_CS_IDS, ALIGN_CS_BTN_IDS,
)

# Local imports (page-specific, no cross-package dependencies)
from cjm_transcript_vad_align.html_ids import AlignmentHtmlIds
from cjm_transcript_vad_align.models import VADChunk, AlignmentUrls
from cjm_transcript_vad_align.utils import (
    get_audio_file_boundaries, get_audio_file_count, get_audio_file_position,
)
from cjm_transcript_vad_align.components.vad_card import (
    create_vad_card_renderer
)
from cjm_transcript_vad_align.components.callbacks import (
    generate_align_callbacks_script, ALIGN_AUDIO_CONFIG,
)
from cjm_transcript_vad_align.components.audio_controls import (
    render_align_audio_controls,
)

# Debug flag for alignment rendering tracing (set False in production)
DEBUG_ALIGN_RENDER = False

## Toolbar & Stats

Alignment-specific toolbar with card count selector.
Rendered into the shared chrome area when the alignment column is active.

In [ ]:
#| export
def render_align_toolbar(
    current_speed:float=1.0,  # Current playback speed
    auto_navigate:bool=False,  # Whether auto-navigate is enabled
    speed_url:str="",  # URL for speed-change POST (server persistence)
    oob:bool=False,  # Whether to render as OOB swap
) -> Any:  # Toolbar component
    """Render the alignment toolbar (speed selector + auto-play toggle)."""
    return Div(
        render_align_audio_controls(
            current_speed=current_speed,
            auto_navigate=auto_navigate,
            speed_url=speed_url,
        ),
        id=AlignmentHtmlIds.ALIGNMENT_TOOLBAR,
        cls=combine_classes(flex_display, gap(2), items.center),
        hx_swap_oob="true" if oob else None
    )

In [ ]:
#| export
def render_align_stats(
    chunks:List[VADChunk],  # Current VAD chunks
    oob:bool=False,  # Whether to render as OOB swap
) -> Any:  # Statistics component
    """Render alignment statistics."""
    total = len(chunks)
    total_dur = sum(c.duration for c in chunks) if chunks else 0.0
    file_count = get_audio_file_count(chunks)

    stats_text = f"{total} chunks \u00b7 {total_dur:.1f}s total"
    if file_count > 1:
        stats_text += f" \u00b7 {file_count} files"

    return Div(
        Span(
            stats_text,
            cls=combine_classes(font_size.sm, text_dui.base_content)
        ),
        id=AlignmentHtmlIds.ALIGNMENT_STATS,
        hx_swap_oob="true" if oob else None
    )

#| export
def render_align_source_position(
    chunks:List[VADChunk],  # Current VAD chunks
    focused_index:int=0,  # Currently focused chunk index
    oob:bool=False,  # Whether to render as OOB swap
) -> Any:  # Audio file position indicator (empty if single file)
    """Render audio file position indicator for the focused chunk."""
    file_count = get_audio_file_count(chunks)

    content = ""
    if file_count > 1:
        pos = get_audio_file_position(chunks, focused_index)
        if pos is not None:
            content = f"File {pos} of {file_count}"

    return Span(
        content,
        id=AlignmentHtmlIds.SOURCE_POSITION,
        cls=combine_classes(font_size.sm, text_dui.base_content),
        hx_swap_oob="true" if oob else None,
    )

## Column Body

Renders the alignment column content area: card stack viewport, hidden audio element,
keyboard infrastructure, and JavaScript callbacks.

In [ ]:
#| export
def render_align_column_body(
    chunks:List[VADChunk],  # VAD chunks to display
    focused_index:int,  # Currently focused chunk index
    visible_count:int,  # Number of visible cards in viewport
    card_width:int,  # Card stack width in rem
    urls:AlignmentUrls,  # URL bundle for alignment routes
    kb_system:Any=None,  # Rendered keyboard system (None when KB managed externally)
    audio_urls:Optional[List[str]]=None,  # Audio file URLs for Web Audio API
    should_play_fn:str="",  # Consumer-defined play guard function name (overrides default zone guard)
) -> Any:  # Div with id=COLUMN_CONTENT
    """Render the alignment column content area with card stack viewport."""
    if DEBUG_ALIGN_RENDER:
        print(f"[ALIGN_RENDER] render_align_column_body called")
        print(f"[ALIGN_RENDER] chunks count: {len(chunks)}")
        print(f"[ALIGN_RENDER] audio_urls count: {len(audio_urls) if audio_urls else 0}")

    # Compute audio file boundaries for visual indicators
    boundaries = get_audio_file_boundaries(chunks)

    # Create card renderer callback with boundary info
    card_renderer = create_vad_card_renderer(audio_file_boundaries=boundaries)

    # Build CardStackState for library viewport
    cs_state = CardStackState(
        focused_index=focused_index,
        visible_count=visible_count,
        card_width=card_width,
    )

    # Render viewport from library
    viewport = render_viewport(
        card_items=chunks,
        state=cs_state,
        config=ALIGN_CS_CONFIG,
        ids=ALIGN_CS_IDS,
        urls=urls.card_stack,
        render_card=card_renderer,
        form_input_name="chunk_index",
    )

    # Generate JS: library card stack JS + Web Audio API callbacks
    callbacks_script = generate_align_callbacks_script(
        ids=ALIGN_CS_IDS,
        button_ids=ALIGN_CS_BTN_IDS,
        config=ALIGN_CS_CONFIG,
        urls=urls.card_stack,
        container_id=AlignmentHtmlIds.COLUMN_CONTENT,
        focus_input_id=ALIGN_CS_IDS.focused_index_input,
        should_play_fn=should_play_fn,
    )

    # Keyboard system elements (optional — may be managed at combined-step level)
    kb_elements = ()
    if kb_system is not None:
        kb_elements = (kb_system.script, kb_system.hidden_inputs, kb_system.action_buttons)

    return Div(
        # Card stack viewport
        viewport,

        # Audio URLs hidden input for Web Audio API parallel decode
        render_audio_urls_input(ALIGN_AUDIO_CONFIG, audio_urls or []),

        # Keyboard navigation system (when provided)
        *kb_elements,

        # Hidden action buttons for JS-triggered card stack nav
        render_card_stack_action_buttons(ALIGN_CS_BTN_IDS, urls.card_stack, ALIGN_CS_IDS),

        # JavaScript callbacks (library card stack JS + Web Audio API playback)
        callbacks_script,

        id=AlignmentHtmlIds.COLUMN_CONTENT,
        cls=combine_classes(grow(), min_h(0), overflow.hidden, flex_display, flex_direction.col)
    )

## Footer & Mini-Stats

Render footer content (progress + stats) for the shared chrome area,
and compact mini-stats text for the column header badge.

In [ ]:
#| export
def render_align_footer_content(
    chunks:List[VADChunk],  # Current VAD chunks
    focused_index:int,  # Currently focused chunk index
) -> Any:  # Footer content with progress indicator, source position, and stats
    """Render footer content with progress indicator, source position, and alignment statistics."""
    total_chunks = len(chunks)

    return Div(
        Div(
            render_progress_indicator(focused_index, total_chunks, ALIGN_CS_IDS, label="VAD Chunk"),
            render_align_source_position(chunks, focused_index),
            cls=combine_classes(flex_display, items.center, gap(3)),
        ),
        render_align_stats(chunks),
        id=AlignmentHtmlIds.ALIGNMENT_FOOTER,
        cls=combine_classes(flex_display, justify.between, items.center, gap(4))
    )

In [ ]:
#| export
def render_align_mini_stats_text(
    chunks:List[VADChunk],  # Current VAD chunks
) -> str:  # Compact stats string for column header badge
    """Generate compact stats string for the alignment column header badge."""
    total = len(chunks)
    total_dur = sum(c.duration for c in chunks) if chunks else 0.0
    file_count = get_audio_file_count(chunks)
    text = f"{total} chunks \u00b7 {total_dur:.1f}s"
    if file_count > 1:
        text += f" \u00b7 {file_count} files"
    return text

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()